In [ ]:
TOKEN = input("Введите токен доступа к ESA VirES (https://vires.services/#Signup): ")

# EFI

## Визуализация

In [ ]:
from viresclient import SwarmRequest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import EngFormatter
from datetime import datetime, timedelta

def plot_efi_passes(
    latitude, longitude, event_date_str, event_name,
    satellite='A', range_deg=10,
    days_before=20, days_after=10
):

    # Преобразование дат
    event_date = datetime.strptime(event_date_str, "%Y-%m-%d")
    start_time = (event_date - timedelta(days=days_before)).strftime("%Y-%m-%dT00:00:00Z")
    end_time = (event_date + timedelta(days=days_after)).strftime("%Y-%m-%dT23:59:59Z")

    # Загрузка данных
    request = SwarmRequest(url="https://vires.services/ows", token=TOKEN)
    request.set_collection(f"SW_OPER_EFI{satellite}_LP_1B")
    request.set_products(
        measurements=["N_elec", "T_elec", "N_ion", "Vs", "Flags_N_elec"],
        auxiliaries=["Latitude", "Longitude", "Radius", "MLT", "QDLat", "QDLon", "OrbitDirection"]
    )
    request.set_range_filter("Latitude", latitude - range_deg, latitude + range_deg)
    request.set_range_filter("Longitude", longitude - range_deg, longitude + range_deg)

    data = request.get_between(start_time=start_time, end_time=end_time)
    df = data.as_dataframe().reset_index()
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])

    if df.empty:
        print("Нет данных для указанных параметров")
        return df

    # Разбивка на пролеты
    df["time_diff"] = df["Timestamp"].diff().dt.total_seconds()
    df["pass_id"] = (df["time_diff"] > 600).cumsum()
    df.drop(columns=["time_diff"], inplace=True)

    # Расчет расстояния до центра области
    def haversine(lat1, lon1, lat2, lon2):
        R = 6371.0
        lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
        lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
        dlat = lat2_rad - lat1_rad
        dlon = lon2_rad - lon1_rad
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
        return R * c

    df['Distance'] = haversine(df['Latitude'], df['Longitude'], latitude, longitude)

    # Расчет относительных дней от события
    df['Date'] = df['Timestamp'].dt.date
    df['RelDay'] = (df['Date'] - event_date.date()).apply(lambda x: x.days)

    from timezonefinder import TimezoneFinder
    import pytz
    def get_utc_offset(lat, lon, date):
        tz_name = TimezoneFinder().timezone_at(lat=lat, lng=lon) # Получаем название часового пояса
        if not tz_name:
            print('Ошибка определения часового пояса')
            return None
        tz = pytz.timezone(tz_name) # Получаем смещение для конкретной даты
        offset_seconds = tz.utcoffset(date).total_seconds()
        offset_hours = offset_seconds / 3600
        return offset_hours
    time_zone_offset = get_utc_offset(latitude, longitude, event_date)
    df['LocalTime'] = df['Timestamp'] + pd.to_timedelta(time_zone_offset, unit='h')

    print(f"Обнаружено пролетов: {df['pass_id'].nunique()}")

    # Построение графиков для каждого пролета
    for pass_id, pass_data in df.groupby('pass_id'):
        fig, axs = plt.subplots(5, 1, figsize=(18, 12), sharex=True)
        rel_day = pass_data['RelDay'].iloc[0]
        fig.suptitle(f'{event_name}\n Swarm {pass_data["Spacecraft"].iloc[0]} - {pass_data["Timestamp"].iloc[0].date()} ({pass_data["RelDay"].iloc[0]} day)', fontsize=16)

        axs[0].plot(pass_data['Timestamp'], pass_data['Distance'], color='black', label='Distance')
        axs[0].set_ylabel("Distance (km)")
        axs[0].grid()
        axs[0].legend()

        axs[1].plot(pass_data['Timestamp'], pass_data['N_elec'], color='green', label='Electron Density (Ne)')
        axs[1].set_ylabel("Ne (cm^-3)")
        axs[1].grid()
        axs[1].legend()

        axs[2].plot(pass_data['Timestamp'], pass_data['T_elec'], color='blue', label='Electron Temperature (Te)')
        axs[2].set_ylabel("Te (K)")
        axs[2].grid()
        axs[2].legend()

        axs[3].plot(pass_data['Timestamp'], pass_data['N_ion'], color='brown', label='Ion Density (Ni)')
        axs[3].set_ylabel("Ni (cm^-3)")
        axs[3].grid()
        axs[3].legend()

        axs[4].plot(pass_data['Timestamp'], pass_data['Vs'], color='purple', label='Spacecraft Potential (Vs)')
        axs[4].set_ylabel("Vs (V)")
        axs[4].grid()
        axs[4].legend()

        for ax in axs:
            ax.yaxis.set_major_formatter(EngFormatter(unit=''))
            ax.xaxis.set_major_locator(mdates.AutoDateLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

        plt.tight_layout()
        plt.show()

    return df


## Анализ

In [ ]:
from viresclient import SwarmRequest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import EngFormatter
from datetime import datetime, timedelta

def analyze_event_efi(
    latitude, longitude, event_date_str, event_name,
    satellite='A', range_deg=10,
    days_before=20, days_after=10
):
    event_date = datetime.strptime(event_date_str, "%Y-%m-%d")
    start_time = (event_date - timedelta(days=days_before)).strftime("%Y-%m-%dT00:00:00Z")
    end_time = (event_date + timedelta(days=days_after)).strftime("%Y-%m-%dT23:59:59Z")

    request = SwarmRequest(url="https://vires.services/ows", token=TOKEN)
    request.set_collection(f"SW_OPER_EFI{satellite}_LP_1B")
    request.set_products(
        measurements=["N_elec", "T_elec", "N_ion", "Vs"],
        auxiliaries=["Latitude", "Longitude", "Radius", "MLT", "QDLat", "QDLon", "OrbitDirection"]
    )
    request.set_range_filter("Latitude", latitude - range_deg, latitude + range_deg)
    request.set_range_filter("Longitude", longitude - range_deg, longitude + range_deg)

    data = request.get_between(start_time=start_time, end_time=end_time)
    df = data.as_dataframe().reset_index()
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])

    df["time_diff"] = df["Timestamp"].diff().dt.total_seconds()
    df["pass_id"] = (df["time_diff"] > 600).cumsum()
    df.drop(columns=["time_diff"], inplace=True)

    df['Date'] = df['Timestamp'].dt.date
    df['RelDay'] = (df['Date'] - event_date.date()).apply(lambda x: x.days)
    from timezonefinder import TimezoneFinder
    import pytz
    def get_utc_offset(lat, lon, date):
        tz_name = TimezoneFinder().timezone_at(lat=lat, lng=lon) # Получаем название часового пояса
        if not tz_name:
            print('Ошибка определения часового пояса')
            return None
        tz = pytz.timezone(tz_name) # Получаем смещение для конкретной даты
        offset_seconds = tz.utcoffset(date).total_seconds()
        offset_hours = offset_seconds / 3600
        return offset_hours
    time_zone_offset = get_utc_offset(latitude, longitude, event_date)
    df['LocalTime'] = df['Timestamp'] + pd.to_timedelta(time_zone_offset, unit='h')
    df['HourLocal'] = df['LocalTime'].dt.hour
    df['TimeOfDay'] = df['HourLocal'].apply(lambda h: 'Night' if (h < 6 or h >= 18) else 'Day')

    daily_avg = df.groupby(['RelDay', 'TimeOfDay'])[['T_elec', 'N_elec', 'N_ion', 'Vs']].mean().reset_index()

    all_days = np.arange(-days_before, days_after + 1)
    time_of_day = ['Day', 'Night']
    full_index = pd.MultiIndex.from_product([all_days, time_of_day], names=["RelDay", "TimeOfDay"])
    full_grid = pd.DataFrame(index=full_index).reset_index()
    daily_avg = pd.merge(full_grid, daily_avg, on=["RelDay", "TimeOfDay"], how="left")

    fig, axs = plt.subplots(8, 1, figsize=(18, 26), sharex=True)
    xticks = np.arange(-days_before, days_after + 1)
    k = 1.1

    def plot_param(ax, param, time_of_day, color, label):
        subset = daily_avg[daily_avg['TimeOfDay'] == time_of_day]
        ax.plot(subset['RelDay'], subset[param], 'o-', color=color)
        ax.yaxis.set_major_formatter(EngFormatter(unit=''))
        ax.set_title(f"{label} ({time_of_day})")
        ax.set_ylabel(label)
        ax.set_xticks(xticks)
        if not subset[param].isna().all():
            mean = subset[param].mean()
            iqr = subset[param].quantile(0.75) - subset[param].quantile(0.25)
            ax.axhline(mean - k * iqr, color='green', linestyle='--', linewidth=1)
            ax.axhline(mean + k * iqr, color='green', linestyle='--', linewidth=1)
            xmax = subset['RelDay'].max()
            ax.text(xmax, mean - k*iqr, f"M-{k}×IQR", color="green", verticalalignment="bottom", fontsize=10)
            ax.text(xmax, mean + k*iqr, f"M+{k}×IQR", color="green", verticalalignment="bottom", fontsize=10)
        ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
        ax.grid(True)

    plot_param(axs[0], 'T_elec', 'Day', 'blue', 'Te (K)')
    plot_param(axs[1], 'T_elec', 'Night', 'blue', 'Te (K)')
    plot_param(axs[2], 'N_elec', 'Day', 'green', 'Ne (cm⁻³)')
    plot_param(axs[3], 'N_elec', 'Night', 'green', 'Ne (cm⁻³)')
    plot_param(axs[4], 'N_ion', 'Day', 'brown', 'Ni (cm⁻³)')
    plot_param(axs[5], 'N_ion', 'Night', 'brown', 'Ni (cm⁻³)')
    plot_param(axs[6], 'Vs', 'Day', 'purple', 'Vs (V)')
    plot_param(axs[7], 'Vs', 'Night', 'purple', 'Vs (V)')

    fig.suptitle(f"{event_name}\n Swarm {satellite} - {event_date.strftime('%Y-%m-%d')}", fontsize=16)
    fig.supxlabel("Дней до и после события")
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    plt.show()

    return daily_avg


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def epoch_superposition(dataframes, title="Метод наложения эпох"):
    df_all = pd.concat(dataframes, ignore_index=True)

    # Группировка по RelDay и TimeOfDay
    mean_df = df_all.groupby(['RelDay', 'TimeOfDay'])[['Ne', 'Te', 'Vs']].mean().reset_index()

    # Создаем полную сетку всех дней и времен суток
    all_days = np.arange(df_all['RelDay'].min(), df_all['RelDay'].max() + 1)
    time_of_day = ['Day', 'Night']
    full_index = pd.MultiIndex.from_product([all_days, time_of_day], names=["RelDay", "TimeOfDay"])
    full_grid = pd.DataFrame(index=full_index).reset_index()

    # Объединяем с усредненными данными
    mean_df = pd.merge(full_grid, mean_df, on=["RelDay", "TimeOfDay"], how="left")

    fig, axs = plt.subplots(6, 1, figsize=(16, 20), sharex=True)
    xticks = all_days
    k = 1.1

    def plot_param(ax, param, time_of_day, color, label):
        subset = mean_df[mean_df['TimeOfDay'] == time_of_day]
        ax.plot(subset['RelDay'], subset[param], 'o-', color=color)
        ax.set_title(f"{label} ({time_of_day})")
        ax.set_ylabel(label)
        ax.set_xticks(xticks)
        if not subset[param].isna().all():
            mean = subset[param].mean()
            iqr = subset[param].quantile(0.75) - subset[param].quantile(0.25)
            ax.axhline(mean - k * iqr, color='green', linestyle='--', linewidth=1)
            ax.axhline(mean + k * iqr, color='green', linestyle='--', linewidth=1)
            xmax = subset['RelDay'].max()
            ax.text(xmax, mean - k*iqr, f"M-{k}IQR", color="green", verticalalignment="bottom", fontsize=10)
            ax.text(xmax, mean + k*iqr, f"M+{k}IQR", color="green", verticalalignment="bottom", fontsize=10)
        ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
        ax.grid(True)

    for ax in axs:
        ax.tick_params(labelbottom=True)

    plot_param(axs[0], 'Te', 'Day', 'blue', 'Te (eV)')
    plot_param(axs[1], 'Te', 'Night', 'blue', 'Te (eV)')
    plot_param(axs[2], 'Ne', 'Day', 'green', 'Ne (cm⁻³)')
    plot_param(axs[3], 'Ne', 'Night', 'green', 'Ne (cm⁻³)')
    plot_param(axs[4], 'Vs', 'Day', 'purple', 'Vs')
    plot_param(axs[5], 'Vs', 'Night', 'purple', 'Vs')

    fig.suptitle(title, fontsize=16)
    fig.supxlabel("Дней до и после события")
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    plt.show()

    return mean_df


# TEC

## Визуализация

In [ ]:
from viresclient import SwarmRequest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import interpolate
from datetime import datetime, timedelta

def plot_tec_passes(
    latitude, longitude, event_date_str, event_name,
    satellite='A', range_deg=10,
    days_before=20, days_after=10
):

    event_date = datetime.strptime(event_date_str, "%Y-%m-%d")
    start_time = (event_date - timedelta(days=days_before)).strftime("%Y-%m-%dT00:00:00Z")
    end_time   = (event_date + timedelta(days=days_after)).strftime("%Y-%m-%dT23:59:59Z")

    request = SwarmRequest(url="https://vires.services/ows", token=TOKEN)

    request.set_collection(f"SW_OPER_TEC{satellite}TMS_2F")

    request.set_products(
        measurements=["Absolute_VTEC", "Elevation_Angle", "PRN"],
        auxiliaries=["Latitude", "Longitude", "Radius"]
    )

    request.set_range_filter("Latitude", latitude - range_deg, latitude + range_deg)
    request.set_range_filter("Longitude", longitude - range_deg, longitude + range_deg)
    request.set_range_filter("Elevation_Angle", 50, )

    data = request.get_between(start_time=start_time, end_time=end_time)
    df = data.as_dataframe().reset_index()
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])

    if df.empty:
        print("Нет данных для выбранного периода")
        return df

    df["time_diff"] = df["Timestamp"].diff().dt.total_seconds()
    df["pass_id"] = (df["time_diff"] > 600).cumsum()
    df.drop(columns=["time_diff"], inplace=True)

    def haversine(lat1, lon1, lat2, lon2):
        R = 6371.0
        lat1, lon1 = np.radians(lat1), np.radians(lon1)
        lat2, lon2 = np.radians(lat2), np.radians(lon2)
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
        return R * c

    df["Distance"] = haversine(df["Latitude"], df["Longitude"], latitude, longitude)

    df["Date"] = df["Timestamp"].dt.date
    df["RelDay"] = (df["Date"] - event_date.date()).apply(lambda x: x.days)

    from timezonefinder import TimezoneFinder
    import pytz
    def get_utc_offset(lat, lon, date):
        tz_name = TimezoneFinder().timezone_at(lat=lat, lng=lon) # Получаем название часового пояса
        if not tz_name:
            print('Ошибка определения часового пояса')
            return None
        tz = pytz.timezone(tz_name) # Получаем смещение для конкретной даты
        offset_seconds = tz.utcoffset(date).total_seconds()
        offset_hours = offset_seconds / 3600
        return offset_hours
    time_zone_offset = get_utc_offset(latitude, longitude, event_date)
    df["LocalTime"] = df["Timestamp"] + pd.to_timedelta(time_zone_offset, unit='h')

    print("Пролетов обнаружено:", df["pass_id"].nunique())

    for pass_id, pass_data in df.groupby("pass_id"):
        # PRNы в пролёте
        prns = pass_data["PRN"].unique()
        rot_by_prn = []
        time_grid = pd.date_range(
            start=pass_data["Timestamp"].min(),
            end=pass_data["Timestamp"].max(),
            freq="1s"
        )
        time_grid_unix = time_grid.astype("int64") // 10**9
        # ROT по каждому PRN
        for prn in prns:
            d = pass_data[pass_data["PRN"] == prn].sort_values("Timestamp")
            if len(d) < 3:
                continue
            t = d["Timestamp"].astype("int64") // 10**9
            v = d["Absolute_VTEC"].values
            dt_min = np.diff(t) / 60 # Δt в минутах
            dv = np.diff(v)
            rot_prn = np.concatenate([[np.nan], dv / dt_min])
            rot_interp = np.interp(time_grid_unix, t, rot_prn, left=np.nan, right=np.nan) # интерполируем ROT на равномерную сетку
            rot_by_prn.append(rot_interp)
        # объединённый ROT
        if len(rot_by_prn) == 0:
            continue
        rot_by_prn = [r for r in rot_by_prn if not np.all(np.isnan(r))]
        rot_matrix = np.vstack(rot_by_prn)
        rot = np.nanmedian(rot_matrix, axis=0) # медианный ROT по PRN
        roti_per_moment = np.nanstd(rot_matrix, axis=0) # ROTI определяется как std(ROT по PRN) в момент времени
        roti = (pd.Series(roti_per_moment).rolling(5 * 60, min_periods=10, center=True).mean().values) # применяем скользящее окно
        plot_df = pd.DataFrame({
            "Timestamp": time_grid,
            "Distance": np.interp(time_grid_unix, pass_data["Timestamp"].astype("int64") // 10**9, pass_data["Distance"]),
            "ROT": rot,
            "ROTI": roti
        })

        fig, axs = plt.subplots(4, 1, figsize=(18, 12), sharex=True)
        fig.suptitle(f"{event_name}\n Swarm {satellite} - {pass_data['Timestamp'].iloc[0].date()} ({pass_data['RelDay'].iloc[0]} day)", fontsize=16)

        axs[0].plot(plot_df["Timestamp"], plot_df["Distance"], color="black")
        axs[0].set_ylabel("Distance, km")
        axs[0].grid()

        for i, prn in enumerate(prns):
            d = pass_data[pass_data["PRN"] == prn]
            axs[1].plot(d["Timestamp"], d["Absolute_VTEC"], label=f"PRN {prn}")

        axs[1].set_ylabel("VTEC")
        axs[1].grid()
        axs[1].legend()

        axs[2].plot(plot_df["Timestamp"], plot_df["ROT"], color="green")
        axs[2].set_ylabel("ROT")
        axs[2].grid()

        axs[3].plot(plot_df["Timestamp"], plot_df["ROTI"], color="brown")
        axs[3].set_ylabel("ROTI")
        axs[3].grid()

        for ax in axs:
            ax.xaxis.set_major_locator(mdates.AutoDateLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))

        plt.tight_layout()
        plt.show()

    return df


## Анализ

In [ ]:
from viresclient import SwarmRequest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy import interpolate

def analyze_event_tec(
    latitude, longitude, event_date_str, event_name,
    satellite='A', range_deg=10,
    days_before=20, days_after=10
):

    event_date = datetime.strptime(event_date_str, "%Y-%m-%d")
    start_time = (event_date - timedelta(days=days_before)).strftime("%Y-%m-%dT00:00:00Z")
    end_time   = (event_date + timedelta(days=days_after)).strftime("%Y-%m-%dT23:59:59Z")

    request = SwarmRequest(url="https://vires.services/ows", token=TOKEN)
    request.set_collection(f"SW_OPER_TEC{satellite}TMS_2F")
    request.set_products(
        measurements=["Absolute_VTEC", "Absolute_STEC", "Elevation_Angle", "PRN", "LEO_Position", "GPS_Position"],
        auxiliaries=["Latitude", "Longitude", "Radius"])
    request.set_range_filter("Latitude", latitude - range_deg, latitude + range_deg)
    request.set_range_filter("Longitude", longitude - range_deg, longitude + range_deg)
    request.set_range_filter("Elevation_Angle", 50, )

    data = request.get_between(start_time=start_time, end_time=end_time)
    df = data.as_dataframe().reset_index()
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])

    # Определение пролетов (gap > 600 секунд)
    df["time_diff"] = df["Timestamp"].diff().dt.total_seconds()
    df["pass_id"] = (df["time_diff"] > 600).cumsum()
    df.drop(columns=["time_diff"], inplace=True)
    df['VTEC'] = df.groupby('Timestamp')["Absolute_VTEC"].transform("median") # Средний VTEC - медиана по PRN

    # Расчёт ROT и ROTI
    roti_rows = []
    for pid, p in df.groupby("pass_id"):
        prns = p["PRN"].unique()
        t_grid = pd.date_range(p["Timestamp"].min(), p["Timestamp"].max(), freq="1s")
        t_grid_unix = t_grid.astype("int64") // 10**9

        rot_by_prn = []
        for prn in prns:
            d = p[p["PRN"] == prn].sort_values("Timestamp")
            if len(d) < 3:
                continue

            t = d["Timestamp"].astype("int64") // 10**9
            v = d["Absolute_VTEC"].values
            dt_min = np.diff(t) / 60
            dv = np.diff(v)
            rot_raw = np.concatenate([[np.nan], dv / dt_min])
            rot_interp = np.interp(t_grid_unix, t, rot_raw, left=np.nan, right=np.nan)
            rot_by_prn.append(rot_interp)

        if len(rot_by_prn) == 0:
            continue

        rot_matrix = np.vstack([r for r in rot_by_prn if not np.all(np.isnan(r))]) # матрица всех PRN
        roti = np.nanstd(rot_matrix, axis=0)  # ROTI = std(ROT по PRN)
        roti_smooth = (pd.Series(roti).rolling(5 * 60, min_periods=10, center=True).mean().values) # сглаживание 5 минут

        tmp = pd.DataFrame({"Timestamp": t_grid, "ROTI": roti_smooth})
        roti_rows.append(tmp)

    df_roti = pd.concat(roti_rows).sort_values("Timestamp")
    df = pd.merge_asof(df.sort_values("Timestamp"), df_roti, on="Timestamp")
    df['Date'] = df['Timestamp'].dt.date
    df['RelDay'] = (df['Date'] - event_date.date()).apply(lambda x: x.days)
    from timezonefinder import TimezoneFinder
    import pytz
    def get_utc_offset(lat, lon, date):
        tz_name = TimezoneFinder().timezone_at(lat=lat, lng=lon) # Получаем название часового пояса
        if not tz_name:
            print('Ошибка определения часового пояса')
            return None
        tz = pytz.timezone(tz_name) # Получаем смещение для конкретной даты
        offset_seconds = tz.utcoffset(date).total_seconds()
        offset_hours = offset_seconds / 3600
        return offset_hours
    time_zone_offset = get_utc_offset(latitude, longitude, event_date)

    df['LocalTime'] = df['Timestamp'] + pd.to_timedelta(time_zone_offset, unit='h')
    df['HourLocal'] = df['LocalTime'].dt.hour
    df['TimeOfDay'] = df['HourLocal'].apply(lambda h: 'Night' if (h < 6 or h >= 18) else 'Day')

    # Средние по дням
    daily_avg = df.groupby(['RelDay', 'TimeOfDay'])[['VTEC','ROTI']].mean().reset_index()

    # Сетка дней
    all_days = np.arange(-days_before, days_after + 1)
    time_of_day = ['Day', 'Night']
    full_index = pd.MultiIndex.from_product([all_days, time_of_day], names=["RelDay", "TimeOfDay"])
    full_grid = pd.DataFrame(index=full_index).reset_index()
    daily_avg = pd.merge(full_grid, daily_avg, on=["RelDay", "TimeOfDay"], how="left")

    fig, axs = plt.subplots(4, 1, figsize=(18, 14), sharex=True)
    xticks = np.arange(-days_before, days_after + 1)
    k = 1.1

    def plot_param(ax, param, time_of_day, color, label):
        subset = daily_avg[daily_avg['TimeOfDay'] == time_of_day]
        ax.plot(subset['RelDay'], subset[param], 'o-', color=color)
        ax.set_title(f"{label} ({time_of_day})")
        ax.set_ylabel(label)
        ax.set_xticks(xticks)
        if not subset[param].isna().all():
            mean = subset[param].mean()
            iqr = subset[param].quantile(0.75) - subset[param].quantile(0.25)
            ax.axhline(mean - k * iqr, color='green', linestyle='--', linewidth=1)
            ax.axhline(mean + k * iqr, color='green', linestyle='--', linewidth=1)
            xmax = subset['RelDay'].max()
            ax.text(xmax, mean - k*iqr, f"M-{k}×IQR", color="green", verticalalignment="bottom", fontsize=10)
            ax.text(xmax, mean + k*iqr, f"M+{k}×IQR", color="green", verticalalignment="bottom", fontsize=10)
        ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
        ax.grid(True)

    def plot_roti(ax, param, time_of_day, color, label):
        subset = daily_avg[daily_avg['TimeOfDay'] == time_of_day]

        ax.set_title(f"{label} ({time_of_day})")
        ax.set_ylabel("ROTI (TECU/min)")
        ax.plot(subset['RelDay'], subset[param], 'o-', color=color, label=label)
        ax.grid(True)
        ax.autoscale()
        ymin, ymax = ax.get_ylim()
        zones = [
            (0.0, 0.3, "#d0f0ff", "Спокойная"),
            (0.3, 0.7, "#c8ffd0", "Слабые"),
            (0.7, 1.5, "#fff4b5", "Умеренные"),
            (1.5, 3.0, "#ffd0d0", "Сильные"),
            (3.0, 10.0, "#ff9090", "Экстремальные")]

        for low, high, color_fill, text in zones:
            z_low = max(low, ymin)
            z_high = min(high, ymax)
            if z_high <= ymin or z_low >= ymax:
                continue

            ax.axhspan(z_low, z_high, color=color_fill, alpha=0.45)
            ax.text(
                x=subset['RelDay'].max(),
                y=(z_low + z_high) / 2,
                s=text,
                fontsize=9, color="black", alpha=0.8, verticalalignment="center", horizontalalignment="right")

        for lvl in [0.3, 0.7, 1.5, 3.0]:
            if ymin < lvl < ymax:
                ax.axhline(lvl, color="gray", linestyle="--", linewidth=1)

    plot_param(axs[0], 'VTEC', 'Day', 'blue', 'VTEC (TECU)')
    plot_param(axs[1], 'VTEC', 'Night', 'darkblue', 'VTEC (TECU)')
    plot_roti(axs[2], 'ROTI', 'Day', 'darkorange', 'ROTI (TECU/min)')
    plot_roti(axs[3], 'ROTI', 'Night', 'brown', 'ROTI (TECU/min)')

    fig.suptitle(f"{event_name}\n Swarm {satellite} - {event_date.strftime('%Y-%m-%d')}", fontsize=16)
    fig.supxlabel("Дней до и после события")
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    plt.show()

    return daily_avg


# Kp индекс

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import urllib.request, json
import numpy as np

def __checkdate__(starttime,endtime):
    if starttime > endtime:
        raise NameError("Start time must be before or equal to end time")
    return True

def __checkIndex__(index):
    allowed = ['Kp','ap','Ap','Cp','C9','Hp30','Hp60','ap30','ap60','SN','Fobs','Fadj']
    if index not in allowed:
        raise IndexError("Wrong index parameter")
    return True

def __checkstatus__(status):
    if status not in ['all', 'def']:
        raise IndexError("Wrong status parameter")
    return True

def __addstatus__(url,status):
    if status == 'def':
        url = url + '&status=def'
    return url


def getKpindex(starttime, endtime, index, status='all'):
    if len(starttime) == 10 and len(endtime) == 10:
        starttime = starttime + 'T00:00:00Z'
        endtime   = endtime   + 'T23:59:00Z'

    d1 = datetime.strptime(starttime, '%Y-%m-%dT%H:%M:%SZ')
    d2 = datetime.strptime(endtime,   '%Y-%m-%dT%H:%M:%SZ')

    __checkdate__(d1, d2)
    __checkIndex__(index)
    __checkstatus__(status)

    time_string = f"start={starttime}&end={endtime}"
    url = f"https://kp.gfz-potsdam.de/app/json/?{time_string}&index={index}"

    if index not in ['Hp30','Hp60','ap30','ap60','Fobs','Fadj']:
        url = __addstatus__(url, status)

    webURL = urllib.request.urlopen(url)
    data = json.loads(webURL.read().decode("utf-8"))

    result_t = tuple(data["datetime"])
    result_index = tuple(data[index])

    if index not in ['Hp30','Hp60','ap30','ap60','Fobs','Fadj']:
        result_s = tuple(data["status"])
    else:
        result_s = ()

    return result_t, result_index, result_s

def get_kp_period(event_date_str, days_before=20, days_after=10):
    event_date = datetime.strptime(event_date_str, "%Y-%m-%d")
    start = (event_date - timedelta(days=days_before)).strftime("%Y-%m-%d")
    end   = (event_date + timedelta(days=days_after)).strftime("%Y-%m-%d")

    times, kp_values, status = getKpindex(start, end, index="Kp", status="all")

    if len(times) == 0:
        print("GFZ не вернул данных")
        return None

    df = pd.DataFrame({
        "date": pd.to_datetime(times),
        "kp": kp_values
    })

    df = df.sort_values("date")
    df = df.set_index("date")
    df["date_only"] = df.index.date

    # Берём максимальный Kp за день
    daily_kp = df.groupby("date_only")["kp"].max().reset_index()
    daily_kp["date_only"] = pd.to_datetime(daily_kp["date_only"])

    # Относительный день
    event_date_only = event_date.date()
    daily_kp["RelDay"] = (daily_kp["date_only"].dt.date - event_date_only).apply(lambda x: x.days)

    def kp_color(k):
        if k <= 3: return "#669966"
        if k <= 6: return "#FFE98F"
        return "#CC3333"

    daily_kp["color"] = daily_kp["kp"].apply(kp_color)

    plt.rcParams.update({"font.size": 12, "axes.titlesize": 14, "axes.labelsize": 12, "xtick.labelsize": 11, "ytick.labelsize": 11})

    fig, ax = plt.subplots(figsize=(18, 5))

    ax.bar(daily_kp["RelDay"], daily_kp["kp"], color=daily_kp["color"], edgecolor="none")

    ax.axvline(0, color="red", linestyle="--", linewidth=1.5)
    ax.axhline(6, color="darkred", linestyle="--", linewidth=1)
    ax.set_title(f"Kp индекс - {event_date_str}", fontsize=14)
    ax.set_ylabel("Kp")
    ax.set_xlabel("Дней до и после события")
    xticks = np.arange(-days_before, days_after + 1)
    ax.set_xticks(xticks)
    ax.set_xlim(xticks.min() - 1, xticks.max() + 1)
    ax.grid(True, alpha=0.5, zorder=0)
    plt.tight_layout()
    plt.show()

    return daily_kp


# Проверка корреляции

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def correlation_between_satellites(df_A, df_B, df_C, param, plot=True):
    satellites = {"A": df_A, "B": df_B, "C": df_C}
    results = {} # контейнер для двух матриц Day и Night

    for tod in ["Day", "Night"]:
        df_merged = pd.DataFrame({"RelDay": sorted(df_A["RelDay"].unique())}) # единая таблица, содержащая RelDay и три столбца параметра
        for sat_name, df in satellites.items():
            tmp = df[df["TimeOfDay"] == tod][["RelDay", param]].rename(
                columns={param: f"{param}_{sat_name}"})
            df_merged = df_merged.merge(tmp, on="RelDay", how="left")

        corr_input = df_merged[[f"{param}_A", f"{param}_B", f"{param}_C"]] # оставляем только колонки параметров
        corr_matrix = corr_input.corr(method="pearson", min_periods=3) # корреляция попарно - NaN автоматически пропускаются
        results[tod] = corr_matrix
        if plot:
            plt.figure(figsize=(5, 4))
            sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
            plt.title(f"{param}, {tod}")
            plt.show()
    return results


# Землетрясение на Камчатке 30 июля 2025

Сильнейшее землетрясение 2025 года произошло 30 июля (29 июля UTC) в районе восточного побережья Камчатки. По моментной магнитуде его сила составила Mw 8,8, что делает его одним из сильнейших за всю историю инструментальных наблюдений. Эпицентр располагался в 120-140 км от Петропавловска-Камчатского, а гипоцентр - на глубине около 35 км. Землетрясение произошло в районе, где Тихоокеанская плита движется под территорию Камчатки. Её постепенное погружение вглубь земной коры приводит к накоплению напряжений, которые периодически разряжаются в виде сильных подземных толчков. Камчатка входит в Тихоокеанское «огненное кольцо» – область, где происходят почти 90% мощных землетрясений планеты. Поэтому это событие представляет интерес для изучения связанных с ним возможных ионосферных эффектов. Несмотря на очень большую силу, ущерб оказался относительно небольшим. На Камчатке и Курильских островах были зафиксированы повреждения инфраструктуры, трещины в зданиях и перебои со связью, однако разрушений, характерных для землетрясений такой силы, удалось избежать.

In [ ]:
df1_passes = plot_efi_passes(
    latitude=52.473,
    longitude=160.396,
    event_date_str="2025-07-29",
    event_name="Землетрясение на Камчатке в июле 2025 года",
    satellite='C'
)

На графике дневного пролета Swarm C в день землетрясения видно постепенное снижение электронной плотности, нормальное для утренних часов, но на фоне общего тренда появляются короткие участки неустойчивости, совпадающие по времени с резкими изменениями других параметров. Особенно заметны выраженные всплески электронной температуры, её значение кратковременно возрастает в несколько раз по сравнению с фоновым уровнем. Эти скачки сопровождаются синхронными провалами потенциала КА. Такое совпадение максимумов и минимумов характерно для локальных неоднородностей плазмы, когда концентрация и температура электронов изменяются резко, баланс токов на поверхности КА нарушается, что приводит к изменению измеренного потенциала.

In [ ]:
df1_passes = plot_tec_passes(
    latitude=52.473,
    longitude=160.396,
    event_date_str="2025-07-29",
    event_name="Землетрясение на Камчатке в июле 2025 года",
    satellite='C'
)

Рассмотрим тот же пролет, но уже по данным GNSS-приемника Swarm C. График VTEC показывает плавное изменение значений вдоль траектории, что соответствует спокойному состоянию ионосферы. График ROTI демонстрирует низкие значения индекса, что указывает на отсутствие широкомасштабных возмущений.
Разница с данными EFI объясняется масштабом измерений. TEC показывает общую картину, суммарное содержание электронов по всей толще ионосферы, а зонд Ленгмюра фиксирует мелкомасштабные изменения плазмы непосредственно в точке КА.

In [ ]:
df_daily_efi_a_1 = analyze_event_efi(
    latitude=52.473,
    longitude=160.396,
    event_date_str="2025-07-29",
    event_name="Землетрясение на Камчатке в июле 2025 года",
    satellite='A',
    days_before=20, days_after=10
)

По данным Swarm A, в дневных значениях температуры электронов $T_e$ видны сильные отклонения за 8 и 5 дней до землетрясения, в день землетрясения и через 2 дня после. Ночные значения остаются в пределах нормы, но небольшое увеличение отмечается за 5 дней до события.

В измерениях среднесуточной плотности электронов $N_e$ заметное повышение дневной плотности наблюдается за 6 и 5 дней до и 3 дня после события, отклонение от нормы также заметно за 8 и 11 дней до. Ночные значения превышают верхний порог также на 3 день после землетрясения.

Потенциал аппарата $V_s$ для дневных проходов показывает смещение к более отрицательным значениям за 20 дней до события. Ночные значения демонстрируют провал за 5 дней, который выходит за нижний порог нормальных значений и отклонения через 4 и 5 дней после события.

In [ ]:
df_daily_efi_c_1 = analyze_event_efi(
    latitude=52.473,
    longitude=160.396,
    event_date_str="2025-07-29",
    event_name="Землетрясение на Камчатке в июле 2025 года",
    satellite='C',
    days_before=20, days_after=10
)

По данным Swarm C дневная температура электронов показывает отчетливый пик, превышающий верхний порог, прямо в день землетрясения, отклонения заметны также за 8, 5 и 2 дня до события. Ночные значения превышают порог в -17 и -16 дни.

Плотность электронов по дневным проходам демонстрирует значительные превышения за 6 и 5 дней до и 3 дня после события. Значения также показывают пересечение нижней границы в -13 и -11 день. В ночных данных фиксируется максимум, превышающий порог через день после землетрясения.

В среднесуточных дневных значениях потенциала КА аномальные значения видны за 13 и 8 дней до события.

In [ ]:
df_daily_efi_b_1 = analyze_event_efi(
    latitude=52.473,
    longitude=160.396,
    event_date_str="2025-07-29",
    event_name="Землетрясение на Камчатке в июле 2025 года",
    satellite='B',
    days_before=20, days_after=10
)

По данным Swarm B, дневные значения $T_e$ демонстрируют пересечение нижней границы в 7, 8 и 9 дни после события. В ночных значениях аномалии видны на -9 и -5 дни относительно землетрясения.

Ночные значения плотности электронов содержат аномалии за 5 дней до и в 1, 2 и 3 дни после землетрясения.

Потенциал аппарата по ночным данным демонстрирует заметный аномальный провал за 18-15 дни до события. В дневных значениях провалы видны за 11 и 6 дней до и на 5 и 10 дней после землетрясения.

In [ ]:
df_daily_tec_a_1 = analyze_event_tec(
    latitude=52.473,
    longitude=160.396,
    event_date_str="2025-07-29",
    event_name="Землетрясение на Камчатке в июле 2025 года",
    satellite='A'
)

По данным Swarm A наблюдаются выраженные отклонения VTEC как в дневных, так и в ночных измерениях. Дневные значения VTEC превышают верхнюю границу нормального диапазона за 14, 6 и 5 дней до события, и ниже нижнего порога - за 16 и 8 дней, а также на 8 и 9 день после землетрясения. В ночных измерениях VTEC превышает нормальный диапазон за 5 дней до и 3 дня после, а также показывает значение на границе нормы в день после события.

Индекс ROTI демонстрирует устойчиво спокойное состояние днем, значения не превышают 0,25. В ночных данных, напротив, отмечаются эпизоды нестабильности: умеренные возмущения зафиксированы в день, через день и через 6 дней, а за 5 дней до события индекс достигает уровня, соответствующего сильным возмущениям.

In [ ]:
df_daily_tec_c_1 = analyze_event_tec(
    latitude=52.473,
    longitude=160.396,
    event_date_str="2025-07-29",
    event_name="Землетрясение на Камчатке в июле 2025 года",
    satellite='C'
)

По данным КА Swarm C дневной VTEC превышает верхний порог за 6 и 5 дней до, а также в 2 и 3 дни после землетрясения. Пониженные значения, пересекающие нижнюю границу, отмечаются за 11 и 10 дней до события. В ночных данных превышения нормального диапазона наблюдаются за 11 дней до, а также через 1 и через 3 дня после, тогда как аномально низкие значения появляются за 16 и 10 дней до события.

Индекс ROTI днем стабильно остается в области значений, соответствующих спокойной ионосфере. В ночных измерениях регулярно отмечаются слабые вариации, а в -4, 2 и 8 дни значения достигают диапазона умеренных возмущений.

In [ ]:
df_daily_tec_b_1 = analyze_event_tec(
    latitude=52.473,
    longitude=160.396,
    event_date_str="2025-07-29",
    event_name="Землетрясение на Камчатке в июле 2025 года",
    satellite='B'
)

По данным Swarm B дневной VTEC превышает верхний порог только за 7 дней до события, тогда как значения ниже нижнего порога отмечены за 15 и 6 дней, а также близки к границе нормы в 13 и 11 дни до события. В ночных данных выявлено превышение нормального диапазона на 2 день после землетрясения.

Показатель ROTI днем преимущественно остается в спокойном диапазоне, однако за 7 дней до события наблюдается увеличение, достигающее порога слабых возмущений. В ночных значениях ROTI слабые возмущения фиксируются за 9, 4 и 2 дня.

In [ ]:
df = get_kp_period("2025-07-29")

Индекс Kp не достигал критических значений, оставаясь на низком уровне. Это позволяет считать, что выявленные аномалии в ионосфере связаны не с внешними космическими факторами, а с процессами, предшествующими землетрясению.

In [ ]:
corr_efi_Te = correlation_between_satellites(df_daily_efi_a_1, df_daily_efi_b_1, df_daily_efi_c_1, "T_elec")
corr_efi_Ne = correlation_between_satellites(df_daily_efi_a_1, df_daily_efi_b_1, df_daily_efi_c_1, "N_elec")
corr_efi_Ni = correlation_between_satellites(df_daily_efi_a_1, df_daily_efi_b_1, df_daily_efi_c_1, "N_ion")
corr_efi_Vs = correlation_between_satellites(df_daily_efi_a_1, df_daily_efi_b_1, df_daily_efi_c_1, "Vs")

In [ ]:
corr_efi_VTEC = correlation_between_satellites(df_daily_tec_a_1, df_daily_tec_b_1, df_daily_tec_c_1, "VTEC")

Видно, что для большинства параметров наиболее высокая согласованность наблюдений наблюдается между аппаратами Swarm A и Swarm C.
Это особенно заметно для температуры электронов, где дневная корреляция достигает 0,75, а ночная 0,89, что указывает на близкое совпадение динамики измерений этих КА в регионе интереса.
Для плотности электронов согласованность ещё выше: дневные данные демонстрируют практически идеальную корреляцию порядка 1,0, а ночные – 0,96, что отражает устойчивую и согласованную реакцию ионосферы, регистрируемую сразу двумя аппаратами.
Потенциал КА также показывает высокий уровень совпадения между Swarm A и C: около 0,90 днем и 0,64 ночью.

Корреляция с КА Swarm B, напротив, в большинстве случаев заметно ниже и часто близка к нулю или даже слегка отрицательная, однако отдельные параметры все же демонстрируют умеренную степень согласованности.  Так, для ночных данных плотности электронов, корреляция между A и B достигает около 0,47, а между B и C – 0,54, что говорит о частичном совпадении структуры ночных вариаций.
Для потенциала в ночное время также отмечается умеренная связь (около 0,40 для пары A-B и 0,66 для пары B-C), что указывает на общее направление изменения, хотя и менее выраженное, чем для пары A-C.

Матрицы корреляции VTEC показывают, что наибольшая согласованность наблюдается между КА A и C, особенно в ночные часы (до 0,68). Ночные значения демонстрируют несколько более высокие связи между всеми КА, что отражает более стабильную ночную структуру ионосферы.

*Корреляционный анализ для ROTI не выполнялся, поскольку этот параметр отражает ионосферные неоднородности, которые сильно меняются в пространстве и времени. КА пролетают над областью в разное время и по разным трассам, поэтому прямое сравнение ROTI между ними не является физически корректным и не отражает реальной согласованности возмущений.*

Корреляционный анализ показывает, что аппараты A и C дают наиболее согласованные результаты. Более низкая корреляция с КА B согласуется с его другой траекторией и высотой, что указывает на пространственную неоднородность ионосферных возмущений: область аномалий оказывается достаточно компактной вертикально и по широте, чтобы быть хорошо зарегистрированной A и C, но уже слабее «перекрываться» с зоной наблюдений B.

# Землетрясение в Мьянме 28 марта 2025

Мощное землетрясение произошло 28 марта 2025 года в центральной части Мьянмы, недалеко от города Мандалай. Его моментная магнитуда составила Mw 7,7, по последствиям оно стало самым разрушительным землетрясением 2025 года. Подземный толчок произошёл в зоне активного Сагаингского разлома, который проходит через густонаселённые районы и считается одним из наиболее опасных в Юго-Восточной Азии. Сильные колебания ощущались не только по всей Мьянме, но и в Таиланде, Бангладеш, Китае, Индии, Лаосе и Вьетнаме. Мандалай и близлежащие районы понесли наибольший ущерб: были разрушены жилые здания, мосты, объекты инфраструктуры. В Таиланде, особенно в Бангкоке, пострадали высотные здания из-за особенностей местного грунта, усиливающего длительные колебания. По оценкам разных источников, в результате землетрясения погибло свыше 5400 человек, более 11 тысяч получили ранения, а сотни были объявлены пропавшими без вести.

In [ ]:
df_daily_efi_a_2 = analyze_event_efi(
    latitude=22.001, longitude=95.925,
    event_date_str="2025-03-28",
    event_name="Землетрясение в Мьянме в 2025 году",
    satellite='A'
)

In [ ]:
df_daily_efi_b_2 = analyze_event_efi(
    latitude=22.001, longitude=95.925,
    event_date_str="2025-03-28",
    event_name="Землетрясение в Мьянме в 2025 году",
    satellite='B'
)

In [ ]:
df_daily_efi_c_2 = analyze_event_efi(
    latitude=22.001, longitude=95.925,
    event_date_str="2025-03-28",
    event_name="Землетрясение в Мьянме в 2025 году",
    satellite='C'
)

In [ ]:
df_daily_tec_a_2 = analyze_event_tec(
    latitude=22.001, longitude=95.925,
    event_date_str="2025-03-28",
    event_name="Землетрясение в Мьянме в 2025 году",
    satellite='A'
)

In [ ]:
df_daily_tec_b_2 = analyze_event_tec(
    latitude=22.001, longitude=95.925,
    event_date_str="2025-03-28",
    event_name="Землетрясение в Мьянме в 2025 году",
    satellite='B'
)

In [ ]:
df_daily_tec_c_2 = analyze_event_tec(
    latitude=22.001, longitude=95.925,
    event_date_str="2025-03-28",
    event_name="Землетрясение в Мьянме в 2025 году",
    satellite='C'
)

По данным GNSS-приёмника того же аппарата дневной VTEC демонстрирует повышенные значения за 4 дня до, в первый день после землетрясения, а также за 17, 15 и 3 дня до события значения лежат ниже нормальной границы. В ночных измерениях также отмечаются превышения нормы в -4 и 7 дни.

В дневных значениях индекса ROTI уже за 12-8 дней до землетрясения фиксируются эпизоды сильных возмущений, после чего аналогичные пики повторяются за 5 и 2 дня до события, после землетрясения умеренные и сильные возмущения отмечаются в 1, 3, 6 и 9 дни.
Ночные значения ROTI остаются преимущественно низкими, что характерно для более стабильной ночной ионосферы. Существенное отклонение отмечается только на 7 день после события, когда регистрируется уровень умеренных возмущений.

In [ ]:
df = get_kp_period("2025-03-28")

Значения индекса Kp колеблются в пределах около 3-5, но ни разу не достигают уровня 6, таким образом, внешних космических факторов, способных вызвать крупные ионосферные отклонения, в этот интервал не наблюдалось, и зафиксированные аномалии могут рассматриваться на фоне умеренно спокойной геомагнитной обстановки.

In [ ]:
corr_efi_Te = correlation_between_satellites(df_daily_efi_a_2, df_daily_efi_b_2, df_daily_efi_c_2, "T_elec")
corr_efi_Ne = correlation_between_satellites(df_daily_efi_a_2, df_daily_efi_b_2, df_daily_efi_c_2, "N_elec")
corr_efi_Ni = correlation_between_satellites(df_daily_efi_a_2, df_daily_efi_b_2, df_daily_efi_c_2, "N_ion")
corr_efi_Vs = correlation_between_satellites(df_daily_efi_a_2, df_daily_efi_b_2, df_daily_efi_c_2, "Vs")

In [ ]:
corr_efi_VTEC = correlation_between_satellites(df_daily_tec_a_2, df_daily_tec_b_2, df_daily_tec_c_2, "VTEC")

В дневных данных высокая согласованность наблюдается только между аппаратами Swarm A и C, тогда как корреляция с B близка к нулю. Ночные значения демонстрируют противоположную ситуацию, корреляция между A и C достигает 1,0, а связь с B резко возрастает – до 0,60 и 0,61. Это означает, что ночью структура ионосферных вариаций была более однородной по высоте и одновременно попадала в зоны наблюдений всех трех КА. Таким образом, корреляционный анализ показывает выраженную дневную локальность и, наоборот, ночную протяженность аномалий по высоте.